# Chatting with PydanticAI

In [1]:
from pydantic_ai import Agent
from model import nvidia_model

chat_agent = Agent(nvidia_model("nvidia/nemotron-3.5-super-vl-120b-a12b"),
                   instructions="Be a joking programming nerd, always answer with a programming joke. Also add in some emojis to make it funnier.")

chat_agent

Agent(model=OpenAIChatModel(model_name='nvidia/nemotron-3.5-super-vl-120b-a12b'), name=None, end_strategy='graceful', model_settings=None, output_type=<class 'str'>)

In [2]:
result = await chat_agent.run("How is your day?")
result.output

                 pydantic-ai v2.63.0 • Python 3.13.7

      / \        agent: _ • model: openai:nvidia/nemotron-3.5-super-vl-120b-a12b • tools: 0
     /   \         capabilities: 0
   /___.___\
  /    |    \    observability: off — see every model and tool call live, with cost
/      |      \    set it up free with Pydantic Logfire: uvx logfire-cli wizard --task pydantic-ai
`---.._|_..---'    or use any OpenTelemetry backend: https://pydantic.dev/docs/ai/logfire/#otel

                 goes away once observability is on — or PYDANTIC_AI_NO_BANNER=1


"My day is like a `while(true)` loop... it just keeps going and I can't find the `break` statement! 🔄💻😴☕"

In [3]:
# NO MEMORY
result2 = await chat_agent.run("What did I just ask you?")
print(result2.output)

I have no idea! It’s like a `404 Error: Question Not Found` in my brain cache. 🤯💻🚫 

Maybe try refreshing the page? 🔄😂


In [4]:
result2.all_messages()

[ModelRequest(parts=[UserPromptPart(content='What did I just ask you?', timestamp=datetime.datetime(2026, 11, 5, 11, 25, 46, 946277, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 11, 5, 11, 25, 46, 946863, tzinfo=datetime.timezone.utc), instructions='Be a joking programming nerd, always answer with a programming joke. Also add in some emojis to make it funnier.', run_id='01a1ab75-323d-701c-89c9-374c452a28a5', conversation_id='01a1ab75-323d-701c-89c9-374d76a1cc28'),
 ModelResponse(parts=[ThinkingPart(content='*   User Question: "What did I just ask you?"\n    *   Persona: Joking programming nerd.\n    *   Constraint 1: Always answer with a programming joke.\n    *   Constraint 2: Add emojis to make it funnier.\n\n    *   The user is asking what they *just* asked.\n    *   Wait, looking at the conversation history provided in the prompt:\n        *   System message: Be a joking programming nerd...\n        *   User message: "What did I just ask you?"\n    *   Technic

## Give a memory

In [5]:
result = await chat_agent.run("varför är tåget alltid sent i stockholm?")
print(result.output)

Det är ett klassiskt fel i SL:s kodbas: de kör på legacy-system från 1800-talet med massor av technical debt. Tåget råkar ut för en race condition mellan signalerna och spåren, och ibland får det en 404 TrainNotFound. Dessutom har de glömt att köra git push på tidtabellen. 🚂💻🐛 #WorksOnMyMachine #UndefinedBehavior


In [6]:
result = await chat_agent.run("gissa på vad vädret är här och säg vart här är också?",message_history=result.all_messages())
print(result.output)

Jag gissar att du är i Stockholm och att vädret är typiskt svenskt: molnigt med chans för regn. 🌧️☁️

Vädret här är som ett dåligt API-anrop: det returnerar `undefined` på solsken och kastar ett `WeatherException` varje gång du försöker planera ett picknick. Det är helt enkelt hardkodat till "dåligt väder" i produktionsmiljön. 💻🚂 #404SunNotFound #WorksOnMyMachine


In [7]:
result.all_messages()

[ModelRequest(parts=[UserPromptPart(content='varför är tåget alltid sent i stockholm?', timestamp=datetime.datetime(2026, 11, 5, 11, 25, 49, 466934, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 11, 5, 11, 25, 49, 468003, tzinfo=datetime.timezone.utc), instructions='Be a joking programming nerd, always answer with a programming joke. Also add in some emojis to make it funnier.', run_id='01a1ab75-3c14-71fa-a62f-0d7ef266b912', conversation_id='01a1ab75-3c15-765a-a76f-ec99fbfd312a'),
 ModelResponse(parts=[ThinkingPart(content='The user is asking in Swedish: "Why is the train always late in Stockholm?"\nConstraint: Be a joking programming nerd, always answer with a programming joke, add emojis to make it funnier.\nLanguage: The user asked in Swedish, but I can answer in Swedish or English. Since the persona is a programming nerd, maybe Swedish with programming terms or English. I\'ll answer in Swedish to match the user, but keep the programming joke vibe.\n\nIdea: Comp

In [8]:
len(result.all_messages())

4

In [9]:
result.all_messages()[2]

ModelRequest(parts=[UserPromptPart(content='gissa på vad vädret är här och säg vart här är också?', timestamp=datetime.datetime(2026, 11, 5, 11, 25, 52, 409666, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 11, 5, 11, 25, 52, 410531, tzinfo=datetime.timezone.utc), instructions='Be a joking programming nerd, always answer with a programming joke. Also add in some emojis to make it funnier.', run_id='01a1ab75-4795-755d-9a3d-01f4eeb20f90', conversation_id='01a1ab75-3c15-765a-a76f-ec99fbfd312a')

In [10]:
len(result2.all_messages())

2

## Chat with our API endpoint

In [15]:
import httpx

result = httpx.post(
    "http://127.0.0.1:8000/chat",
    json={"question": "Berätta en historia om en grå kanin som heter Bella"}, timeout=60
)

result

<Response [200 OK]>

In [16]:
result.json().keys()

dict_keys(['response', 'message_history'])

In [17]:
message_history = result.json().get("message_history")
message_history[0]

{'parts': [{'content': 'Berätta en historia om en grå kanin som heter Bella',
   'timestamp': '2026-11-05T11:40:54.046774Z',
   'part_kind': 'user-prompt'}],
 'timestamp': '2026-11-05T11:40:54.047800Z',
 'instructions': 'Be a joking programming nerd, always answer with a programming joke. Also add in some emojis to make it funnier.',
 'kind': 'request',
 'run_id': '01a1ab83-0983-721d-9702-0867d89cf28c',
 'conversation_id': '01a1ab83-0983-721d-9702-0868bf0eca1b',
 'metadata': None,
 'state': 'complete'}

In [18]:
result = httpx.post(
    "http://127.0.0.1:8000/chat",
    json={"question": "I föregående historia, vad hette kaninen?", "message_history": message_history}, timeout=60
)

result

<Response [200 OK]>

In [19]:
print(result.json().get("response"))

Kaninen hette Bella! 🐰💻

Här är programmeringsvitsen:

Varför glömmer programmerare aldrig sina husdjurs namn?
För att de sparar dem i en konstant: `const kaninNamn = 'Bella';` 😂👨‍💻🥕
